In [2]:
import ee
import pandas as pd
import os

credentials = ee.ServiceAccountCredentials(
    email='agroalert-service@ee-ishmaelfc.iam.gserviceaccount.com',
    key_file='C:/Users/ELITE/Documents/AGROALERT/service_account.json'
)
ee.Initialize(credentials)

communities = [
    {"name": "Tamale",     "region": "Northern",    "lat": 9.4008,  "lon": -0.8393},
    {"name": "Techiman",   "region": "Brong-Ahafo", "lat": 7.5833,  "lon": -1.9333},
    {"name": "Kumasi",     "region": "Ashanti",     "lat": 6.6885,  "lon": -1.6244},
    {"name": "Ho",         "region": "Volta",       "lat": 6.6000,  "lon":  0.4700},
    {"name": "Bolgatanga", "region": "Upper East",  "lat": 10.7856, "lon": -0.8514},
]

def get_lst(community, start_date, end_date):
    point = ee.Geometry.Point([community['lon'], community['lat']])
    collection = (ee.ImageCollection('MODIS/061/MOD11A2')
        .filterBounds(point)
        .filterDate(start_date, end_date)
        .select('LST_Day_1km'))

    def extract(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point.buffer(10000),
            scale=1000
        )
        raw = val.get('LST_Day_1km')
        # Only convert if value exists
        lst = ee.Algorithms.If(
            raw,
            ee.Number(raw).multiply(0.02).subtract(273.15),
            None
        )
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'lst_celsius': lst,
            'community': community['name'],
            'region': community['region']
        })
    return collection.map(extract)

all_features = ee.FeatureCollection([])
for community in communities:
    fc = get_lst(community, '2022-01-01', '2023-12-31')
    all_features = all_features.merge(fc)

print("Fetching LST from MODIS...")
data = all_features.getInfo()

records = []
for feature in data['features']:
    props = feature['properties']
    if props.get('lst_celsius') is not None:
        records.append({
            'community': props.get('community'),
            'region': props.get('region'),
            'date': props.get('date'),
            'lst_celsius': props.get('lst_celsius')
        })

df_soil = pd.DataFrame(records)
df_soil['date'] = pd.to_datetime(df_soil['date'])
df_soil = df_soil.sort_values(['community', 'date']).reset_index(drop=True)
df_soil.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/soil_moisture_raw.csv', index=False)

print(f"Total records: {len(df_soil)}")
print(df_soil.head(10))

Fetching LST from MODIS...
Total records: 367
    community      region       date  lst_celsius
0  Bolgatanga  Upper East 2022-01-01    35.620115
1  Bolgatanga  Upper East 2022-01-09    37.958092
2  Bolgatanga  Upper East 2022-01-17    35.993902
3  Bolgatanga  Upper East 2022-01-25    35.619920
4  Bolgatanga  Upper East 2022-02-02    35.531512
5  Bolgatanga  Upper East 2022-02-10    41.056204
6  Bolgatanga  Upper East 2022-02-18    39.613532
7  Bolgatanga  Upper East 2022-02-26    38.172127
8  Bolgatanga  Upper East 2022-03-06    42.042138
9  Bolgatanga  Upper East 2022-03-14    34.309504


In [3]:
import ee
import pandas as pd

credentials = ee.ServiceAccountCredentials(
    email='agroalert-service@ee-ishmaelfc.iam.gserviceaccount.com',
    key_file='C:/Users/ELITE/Documents/AGROALERT/service_account.json'
)
ee.Initialize(credentials)

communities = [
    {"name": "Tamale",           "region": "Northern",      "lat": 9.4008,  "lon": -0.8393},
    {"name": "Techiman",         "region": "Bono East",     "lat": 7.5833,  "lon": -1.9333},
    {"name": "Kumasi",           "region": "Ashanti",       "lat": 6.6885,  "lon": -1.6244},
    {"name": "Ho",               "region": "Volta",         "lat": 6.6000,  "lon":  0.4700},
    {"name": "Bolgatanga",       "region": "Upper East",    "lat": 10.7856, "lon": -0.8514},
    {"name": "Wa",               "region": "Upper West",    "lat": 10.0601, "lon": -2.5099},
    {"name": "Sunyani",          "region": "Bono",          "lat": 7.3349,  "lon": -2.3123},
    {"name": "Koforidua",        "region": "Eastern",       "lat": 6.0940,  "lon": -0.2591},
    {"name": "Cape Coast",       "region": "Central",       "lat": 5.1053,  "lon": -1.2466},
    {"name": "Sefwi Wiawso",     "region": "Western North", "lat": 6.2069,  "lon": -2.4856},
    {"name": "Damongo",          "region": "Savannah",      "lat": 9.0833,  "lon": -1.8167},
    {"name": "Nalerigu",         "region": "North East",    "lat": 10.5167, "lon": -0.3667},
    {"name": "Dambai",           "region": "Oti",           "lat": 8.0667,  "lon":  0.1833},
    {"name": "Goaso",            "region": "Ahafo",         "lat": 6.8017,  "lon": -2.5181},
    {"name": "Sekondi-Takoradi", "region": "Western",       "lat": 4.9347,  "lon": -1.7137},
]

def get_lst(community, start_date, end_date):
    point = ee.Geometry.Point([community['lon'], community['lat']])
    collection = (ee.ImageCollection('MODIS/061/MOD11A2')
        .filterBounds(point)
        .filterDate(start_date, end_date)
        .select('LST_Day_1km'))
    def extract(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point.buffer(10000),
            scale=1000
        )
        raw = val.get('LST_Day_1km')
        lst = ee.Algorithms.If(
            raw,
            ee.Number(raw).multiply(0.02).subtract(273.15),
            None
        )
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'lst_celsius': lst,
            'community': community['name'],
            'region': community['region']
        })
    return collection.map(extract)

print("Fetching MODIS LST for 15 communities...")
all_features = ee.FeatureCollection([])
for i, community in enumerate(communities):
    fc = get_lst(community, '2022-01-01', '2023-12-31')
    all_features = all_features.merge(fc)
    print(f"  Queued {i+1}/15: {community['name']}")

print("\nFetching from GEE servers...")
data = all_features.getInfo()

records = []
for feature in data['features']:
    props = feature['properties']
    if props.get('lst_celsius') is not None:
        records.append({
            'community': props.get('community'),
            'region': props.get('region'),
            'date': props.get('date'),
            'lst_celsius': props.get('lst_celsius')
        })

df_lst = pd.DataFrame(records)
df_lst['date'] = pd.to_datetime(df_lst['date'])
df_lst = df_lst.sort_values(['community','date']).reset_index(drop=True)
df_lst.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/soil_moisture_raw.csv', index=False)

print(f"\nTotal LST records: {len(df_lst)}")
print(f"Communities: {df_lst['community'].nunique()}")
print(df_lst['community'].value_counts())

Fetching MODIS LST for 15 communities...
  Queued 1/15: Tamale
  Queued 2/15: Techiman
  Queued 3/15: Kumasi
  Queued 4/15: Ho
  Queued 5/15: Bolgatanga
  Queued 6/15: Wa
  Queued 7/15: Sunyani
  Queued 8/15: Koforidua
  Queued 9/15: Cape Coast
  Queued 10/15: Sefwi Wiawso
  Queued 11/15: Damongo
  Queued 12/15: Nalerigu
  Queued 13/15: Dambai
  Queued 14/15: Goaso
  Queued 15/15: Sekondi-Takoradi

Fetching from GEE servers...

Total LST records: 1150
Communities: 15
community
Nalerigu            86
Bolgatanga          84
Cape Coast          84
Dambai              84
Damongo             83
Wa                  83
Sekondi-Takoradi    82
Koforidua           78
Techiman            77
Ho                  74
Sefwi Wiawso        74
Tamale              70
Goaso               65
Sunyani             64
Kumasi              62
Name: count, dtype: int64
